# A1. AI 辅助选品：竞品 Review 分析实战

> 配套模块：[A1 选品与市场洞察](../paths/a-operators/a1-product-research.md)
>
> 本 Notebook 演示如何用 AI 批量分析竞品 Review，提取痛点和机会点。
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-skills/blob/main/notebooks/a1-product-research.ipynb)

## 1. 准备工作

本 Notebook 不需要 API Key，使用纯 Python 进行数据处理和分析。
AI 分析部分提供 Prompt 模板，你可以复制到 ChatGPT/Claude 中使用。

In [ ]:
import pandas as pd
from collections import Counter
import re

print('环境准备完成')

## 2. 加载竞品 Review 数据

实际使用时，你可以从 Amazon 前台手动复制 Review，或用 Helium 10 等工具导出。

In [ ]:
from pathlib import Path

REVIEW_CSV = Path('reviews.csv')
if not REVIEW_CSV.is_file():
    raise FileNotFoundError(
        '请把真实 Review 导出为 reviews.csv；必需列：rating, text。'
    )

df = pd.read_csv(REVIEW_CSV)
required_columns = {'rating', 'text'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f'Review CSV 缺少必需列: {sorted(missing_columns)}')
if df.empty:
    raise ValueError('Review CSV 不能为空')
df['rating'] = pd.to_numeric(df['rating'], errors='raise')
if not df['rating'].between(1, 5).all():
    raise ValueError('rating 必须在 1 到 5 之间')
df['text'] = df['text'].fillna('').astype(str).str.strip()
if (df['text'] == '').any():
    raise ValueError('text 不能为空')
print(f'共 {len(df)} 条 Review')
print(f'评分分布：\n{df["rating"].value_counts().sort_index()}')

## 3. 基础数据分析（Python）

In [ ]:
# 评分统计
avg_rating = df['rating'].mean()
positive = len(df[df['rating'] >= 4])
negative = len(df[df['rating'] <= 2])
neutral = len(df[df['rating'] == 3])

print(f'平均评分: {avg_rating:.1f}/5')
print(f'好评(4-5星): {positive} ({positive/len(df)*100:.0f}%)')
print(f'差评(1-2星): {negative} ({negative/len(df)*100:.0f}%)')
print(f'中评(3星): {neutral} ({neutral/len(df)*100:.0f}%)')

In [ ]:
# 关键词频率分析
all_text = ' '.join(df['text'].str.lower())
# 提取有意义的词组
keywords = re.findall(r'\b[a-z]{4,}\b', all_text)
# 过滤停用词
stopwords = {'this', 'that', 'with', 'from', 'have', 'been', 'were', 'they', 'their', 'about', 'would', 'could', 'after', 'very', 'much', 'than', 'also', 'just', 'only'}
keywords = [w for w in keywords if w not in stopwords]

print('Top 20 高频关键词：')
for word, count in Counter(keywords).most_common(20):
    print(f'  {word}: {count}')

## 4. AI 深度分析（Prompt 模板）

将以下 Prompt + Review 数据复制到 ChatGPT/Claude 中，获得结构化的痛点分析报告。

In [ ]:
# 生成可以直接粘贴到 AI 的 Prompt
negative_reviews = df[df['rating'] <= 3]['text'].tolist()
positive_reviews = df[df['rating'] >= 4]['text'].tolist()

prompt = f"""你是一个跨境电商选品分析师。请分析以下竞品（便携充电宝）的 Review 数据。

=== 差评和中评（{len(negative_reviews)} 条）===
{chr(10).join(f'- {r}' for r in negative_reviews)}

=== 好评（{len(positive_reviews)} 条）===
{chr(10).join(f'- {r}' for r in positive_reviews)}

请输出：
1. 痛点排名（按频率排序，每个痛点标注出现次数和严重程度）
2. 好评亮点（用户最喜欢的 Top 5 特性）
3. 选品建议（如果我要做一款竞品，应该重点解决哪 3 个痛点）
4. Listing 建议（基于好评亮点，Listing 应该突出哪些卖点）
"""

print('=== 复制以下内容到 ChatGPT/Claude ===')
print(prompt)

## 5. 下一步

- 用更多真实竞品 Review 扩展分析
- 扩展到多个竞品的对比分析
- 结合 [A2 Listing 优化](../paths/a-operators/a2-listing-optimization.md) 生成 Listing 文案
- 查看完整方法论：[A1 选品与市场洞察](../paths/a-operators/a1-product-research.md)